<div style="font-size:2em; font-weight:bold; margin-bottom:8px;">03 — Chunk the Cleaned Corpus</div>

This notebook reads the cleaned SciFact corpus produced by notebook 02 and splits each document into smaller, overlapping **chunks** that are the right size for embedding and retrieval.

It is **Step 3** of the RAG data indexing pipeline — chunking only.

---

**What this notebook does:**
1. Loads the cleaned corpus from `data/processed/02_clean_corpus.jsonl`
2. Explores document lengths so you can see why chunking is needed
3. Defines three chunking strategies: `character`, `recursive`, and `token-aware`
4. Compares the three strategies on a single document
5. Chunks the whole corpus with the configured strategy
6. Inspects the chunk-size distribution
7. Saves the chunks to `data/processed/03_chunks.jsonl`
8. Reads the output back to verify it looks correct

**What this notebook intentionally does NOT do:**
- No metadata enrichment (that is notebook 04)
- No embedding (that is notebook 05)
- No Qdrant indexing

> **Before running:** make sure dependencies are installed.
> ```bash
> pip install -r requirements.txt
> ```
> Chunking uses `langchain-text-splitters` and a Hugging Face tokenizer from `transformers`.

---
## 1. Imports

We need a small set of tools:
- **`json`** — read and write JSON Lines files
- **`statistics`** — quick min / mean / max summaries of chunk sizes
- **`pathlib.Path`** — OS-independent file paths
- **`CharacterTextSplitter`** and **`RecursiveCharacterTextSplitter`** — the two character-based splitters from LangChain
- **`AutoTokenizer`** — loads the embedding model's tokenizer so we can split by real tokens, not characters

In [1]:
# ── [1 / 10] Imports ────────────────────────────────────────────────────────

import json
import statistics
from pathlib import Path

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

print("Imports ready.")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports ready.


---
## 2. Configuration — Strategy and Sizes

All tunable values live in one place. The defaults match `.env.example`
(`chunk_size = 800`, `chunk_overlap = 100`, `strategy = recursive`).

| Setting | Meaning |
|---|---|
| `STRATEGY` | Which splitter to use for the full run: `character`, `recursive`, or `token` |
| `CHUNK_SIZE` | Target chunk size **in characters** (for `character` / `recursive`) |
| `CHUNK_OVERLAP` | How many characters each chunk shares with the previous one |
| `TOKEN_CHUNK_SIZE` | Target chunk size **in tokens** (for the `token` strategy) |
| `TOKEN_CHUNK_OVERLAP` | Token overlap for the `token` strategy |
| `EMBEDDING_MODEL` | Used only to load the matching tokenizer for token-aware chunking |

> **Why two size settings?** Characters and tokens are not the same unit.
> `800` characters is roughly `180–220` tokens for English text, so the token
> strategy uses its own size so the comparison is fair.

In [2]:
# ── [2 / 10] Configuration ──────────────────────────────────────────────────

# Which strategy to use for the full-corpus run
STRATEGY = "recursive"          # one of: "character", "recursive", "token"

# Character-based sizing (character + recursive strategies)
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 100

# Token-based sizing (token strategy)
TOKEN_CHUNK_SIZE    = 200
TOKEN_CHUNK_OVERLAP = 30

# Tokenizer to load for token-aware chunking (matches the embedding model in notebook 05)
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

# Resolve the project root no matter where the notebook is run from
_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

# Input — produced by notebook 02
INPUT_FILE = ROOT / "data" / "processed" / "02_clean_corpus.jsonl"

# Output — chunks for the metadata step
OUTPUT_DIR  = ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "03_chunks.jsonl"

print(f"Project root : {ROOT}")
print(f"Input file   : {INPUT_FILE}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Strategy     : {STRATEGY}")
print(f"Input exists : {INPUT_FILE.exists()}")

Project root : /app
Input file   : /app/data/processed/02_clean_corpus.jsonl
Output file  : /app/data/processed/03_chunks.jsonl
Strategy     : recursive
Input exists : True


---
## 3. Load the Cleaned Corpus

We read the JSONL file produced by notebook 02 into a list of dictionaries.
Each document has the schema: `document_id`, `title`, `text`, `source`, `dataset_config`.

If the input file is missing, we stop with a clear error telling you to run
notebook 02 first.

In [3]:
# ── [3 / 10] Load the Cleaned Corpus ────────────────────────────────────────

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Run notebook 02_clean_corpus.ipynb first."
    )

documents = []
with INPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

print(f"Loaded {len(documents):,} documents")
print()
print("Example document keys:", list(documents[0].keys()))

Loaded 5,183 documents

Example document keys: ['document_id', 'title', 'text', 'source', 'dataset_config']


---
## 4. Explore — Why Do We Need Chunking?

Embedding models have a maximum input length, and retrieval works best when each
stored unit is focused on a single idea. If a document is much longer than the
chunk size, we must split it.

Below we measure the document length distribution (in characters) so we can see
how many documents are longer than our `CHUNK_SIZE` and will therefore be split
into multiple chunks.

In [4]:
# ── [4 / 10] Explore — Document Length Distribution ─────────────────────────

lengths = [len(doc.get("text", "") or "") for doc in documents]

print(f"Documents            : {len(lengths):,}")
print(f"Min length  (chars)  : {min(lengths):,}")
print(f"Mean length (chars)  : {int(statistics.mean(lengths)):,}")
print(f"Max length  (chars)  : {max(lengths):,}")
print()

longer_than_chunk = sum(1 for n in lengths if n > CHUNK_SIZE)
pct = 100 * longer_than_chunk / len(lengths)
print(f"Docs longer than CHUNK_SIZE ({CHUNK_SIZE}) : {longer_than_chunk:,}  ({pct:.1f}%)")
print("These documents will be split into 2 or more chunks.")

Documents            : 5,183
Min length  (chars)  : 174
Mean length (chars)  : 1,400
Max length  (chars)  : 10,000

Docs longer than CHUNK_SIZE (800) : 4,674  (90.2%)
These documents will be split into 2 or more chunks.


---
## 5. Define the Three Chunking Strategies

The project goal asks for three strategies so we can compare their impact later.
We wrap them in a single factory function `build_splitter(strategy)` that returns
a ready-to-use LangChain splitter.

| Strategy | How it splits | When it is useful |
|---|---|---|
| `character` | Cuts every `CHUNK_SIZE` characters, ignoring word boundaries | Simple baseline |
| `recursive` | Tries paragraph → line → sentence → word boundaries first | Best general-purpose default |
| `token` | Splits by real model tokens using the embedding model's tokenizer | Keeps chunks under the model's token limit |

The token tokenizer is loaded lazily (only when needed) because downloading it
takes a few seconds on the first run.

In [5]:
# ── [5 / 10] Define the Three Chunking Strategies ───────────────────────────

# Cache the tokenizer so we load it at most once per session
_tokenizer_cache = {}


def get_tokenizer():
    """Load (and cache) the embedding model's tokenizer for token-aware splitting."""
    if "tok" not in _tokenizer_cache:
        print(f"Loading tokenizer for {EMBEDDING_MODEL} ...")
        _tokenizer_cache["tok"] = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
    return _tokenizer_cache["tok"]


def build_splitter(strategy: str):
    """Return a LangChain splitter for the requested strategy."""
    if strategy == "character":
        return CharacterTextSplitter(
            separator="",
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
        )
    if strategy == "recursive":
        return RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
        )
    if strategy == "token":
        return RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            get_tokenizer(),
            chunk_size=TOKEN_CHUNK_SIZE,
            chunk_overlap=TOKEN_CHUNK_OVERLAP,
        )
    raise ValueError(f"Unknown strategy: {strategy!r}")


print("build_splitter() defined for strategies: character, recursive, token")

build_splitter() defined for strategies: character, recursive, token


---
## 6. Compare the Strategies on One Document

Before processing the whole corpus, we run all three strategies on a single long
document and compare how many chunks each one produces and how big they are.
This makes the trade-offs concrete and helps you understand the configured choice.

In [6]:
# ── [6 / 10] Compare the Strategies on One Document ─────────────────────────

# Pick the longest document so every strategy actually has to split it
sample_doc = max(documents, key=lambda d: len(d.get("text", "") or ""))
sample_text = sample_doc["text"]

print(f"Sample document_id : {sample_doc['document_id']}")
print(f"Sample length      : {len(sample_text):,} chars")
print()

for strategy in ("character", "recursive", "token"):
    chunks = build_splitter(strategy).split_text(sample_text)
    sizes = [len(c) for c in chunks]
    print(f"{strategy:<10} -> {len(chunks):>3} chunks | "
          f"sizes(chars) min/mean/max = "
          f"{min(sizes)}/{int(statistics.mean(sizes))}/{max(sizes)}")

Sample document_id : 10749308
Sample length      : 10,000 chars

character  ->  15 chunks | sizes(chars) min/mean/max = 200/759/800
recursive  ->  15 chunks | sizes(chars) min/mean/max = 225/756/799
Loading tokenizer for BAAI/bge-small-en-v1.5 ...


token      ->  11 chunks | sizes(chars) min/mean/max = 959/1049/1122


---
## 7. Chunk the Whole Corpus

Now we split every document with the configured `STRATEGY`. For each chunk we keep
the original document fields and add a `chunk_index` (0, 1, 2, ... within that
document). The unique `chunk_id` is added in notebook 04 (metadata enrichment).

Empty chunks (which can appear for very short documents) are skipped so they never
reach the index.

In [7]:
# ── [7 / 10] Chunk the Whole Corpus ─────────────────────────────────────────

splitter = build_splitter(STRATEGY)

all_chunks = []
for doc in documents:
    text = doc.get("text", "") or ""
    pieces = splitter.split_text(text)

    chunk_index = 0
    for piece in pieces:
        piece = piece.strip()
        if not piece:
            continue  # never store empty chunks
        all_chunks.append({
            "document_id"    : doc["document_id"],
            "chunk_index"    : chunk_index,
            "text"           : piece,
            "title"          : doc.get("title", ""),
            "source"         : doc.get("source", ""),
            "dataset_config" : doc.get("dataset_config", ""),
        })
        chunk_index += 1

print(f"Documents in  : {len(documents):,}")
print(f"Chunks out    : {len(all_chunks):,}")
print(f"Avg chunks/doc: {len(all_chunks) / len(documents):.2f}")

Documents in  : 5,183
Chunks out    : 12,281
Avg chunks/doc: 2.37


---
## 8. Inspect the Chunk-Size Distribution

A healthy chunking run produces chunks that are mostly close to `CHUNK_SIZE`,
with no empty chunks and no giant outliers. We print the distribution and show
one real example chunk.

In [8]:
# ── [8 / 10] Inspect the Chunk-Size Distribution ────────────────────────────

sizes = [len(c["text"]) for c in all_chunks]

print(f"Total chunks         : {len(sizes):,}")
print(f"Min size  (chars)    : {min(sizes):,}")
print(f"Mean size (chars)    : {int(statistics.mean(sizes)):,}")
print(f"Max size  (chars)    : {max(sizes):,}")
print(f"Empty chunks         : {sum(1 for n in sizes if n == 0)}")
print()
print("Example chunk:")
example_chunk = all_chunks[0]
for key, value in example_chunk.items():
    display = str(value)[:100] + "..." if len(str(value)) > 100 else value
    print(f"  {key:<14} : {display}")

Total chunks         : 12,281
Min size  (chars)    : 93
Mean size (chars)    : 646
Max size  (chars)    : 800
Empty chunks         : 0

Example chunk:
  document_id    : 4983
  chunk_index    : 0
  text           : Alterations of the architecture of cerebral white matter in the developing human brain can affect co...
  title          : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...
  source         : BeIR/scifact
  dataset_config : corpus


---
## 9. Save the Chunks to JSONL

We write all chunks to `data/processed/03_chunks.jsonl`, one chunk per line.
This file is the input to notebook 04, which attaches the retrieval metadata.

In [9]:
# ── [9 / 10] Save the Chunks to JSONL ───────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Saved   : {len(all_chunks):,} chunks")
print(f"Output  : {OUTPUT_FILE.resolve()}")

Saved   : 12,281 chunks
Output  : /app/data/processed/03_chunks.jsonl


---
## 10. Verify the Output File

We read the file back and confirm:
- It contains the expected number of chunks
- Each line is valid JSON with the expected keys
- No chunk is empty
- `chunk_index` restarts at 0 for each new document

In [10]:
# ── [10 / 10] Verify the Output File ────────────────────────────────────────

loaded = []
with OUTPUT_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        loaded.append(json.loads(line))

expected_keys = {"document_id", "chunk_index", "text", "title", "source", "dataset_config"}

empty   = sum(1 for c in loaded if not c["text"].strip())
bad_key = sum(1 for c in loaded if set(c.keys()) != expected_keys)

print(f"Chunks in output file : {len(loaded):,}")
print(f"Empty chunks          : {empty}")
print(f"Rows with wrong keys  : {bad_key}")
print()

assert len(loaded) == len(all_chunks), "chunk count mismatch"
assert empty == 0,   "found empty chunks"
assert bad_key == 0, "found rows with unexpected keys"
print("Chunking verified — output file looks correct.")
print()

print("=" * 60)
print("First 2 chunks from 03_chunks.jsonl")
print("=" * 60)
for i, chunk in enumerate(loaded[:2]):
    print(f"\n--- Chunk {i} ---")
    for key, value in chunk.items():
        display = str(value)[:100] + "..." if len(str(value)) > 100 else value
        print(f"  {key:<14} : {display}")

Chunks in output file : 12,281
Empty chunks          : 0
Rows with wrong keys  : 0

Chunking verified — output file looks correct.

First 2 chunks from 03_chunks.jsonl

--- Chunk 0 ---
  document_id    : 4983
  chunk_index    : 0
  text           : Alterations of the architecture of cerebral white matter in the developing human brain can affect co...
  title          : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...
  source         : BeIR/scifact
  dataset_config : corpus

--- Chunk 1 ---
  document_id    : 4983
  chunk_index    : 1
  text           : at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior lim...
  title          : Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion ten...
  source         : BeIR/scifact
  dataset_config : corpus
